<a href="https://colab.research.google.com/github/vectara/example-notebooks/blob/main/notebooks/api-examples/17-cross-tenant-query-web-get.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cross-Tenant Query with `web_get`

A `corpora_search` tool searches corpora in the agent's own Vectara account. Sometimes the data an agent needs lives in a **different** Vectara account: a partner's or supplier's knowledge base, or a business unit that runs its own account. The owner of that data can give you a **query-only API key** scoped to one corpus. This notebook shows how to let an agent query that corpus safely.

The agent calls the other account's [query API](https://docs.vectara.com/docs/rest-api/query-corpus) with the built-in `web_get` tool, configured so that:

- **The credential never reaches the LLM.** The partner's key is stored as an [agent secret](https://docs.vectara.com/docs/agents/secrets) and injected by `argument_override.auth`. It doesn't appear in the agent config, the prompt, or the audit events.
- **The credential can only go to one place.** `url` and `method` are pinned, so the LLM can't send the key to a different host or endpoint.
- **The LLM only writes the question.** An `input_transform` (a [jq](https://jqlang.github.io/jq/) expression) wraps the LLM's plain-text question in the JSON body the query API expects, and an `output_transform` trims the response to the answer and its sources.

The scenario: **Acme** resells **Globex** hardware. Acme's support agent answers from Acme's own corpus with `corpora_search`, and from Globex's product and policy corpus, which lives in Globex's account, with `web_get`.

This builds on `12-web-get-tool.ipynb` (`argument_override`, one tool per operation) and `15-ref-secrets-and-access-control.ipynb` (secrets and `$ref`).

## Getting Started

All you need is a `VECTARA_API_KEY` that can create corpora, agents, and API keys.

To keep the notebook runnable with one account, Step 2 **simulates** the partner: it creates the Globex corpus in your own account, plus an API key that can only query that corpus. The agent has no other access to it: everything goes through the scoped key over HTTPS, exactly as it would across accounts. With a real partner, you'd skip Step 2 and use the corpus key and query-only API key they give you.

The notebook is self-contained and deletes everything it creates at the end.

## Setup

In [1]:
import os
import json
import time

import requests

api_key = os.environ['VECTARA_API_KEY']

BASE_URL = "https://api.vectara.io/v2"

headers = {
    "x-api-key": api_key,
    "Content-Type": "application/json"
}

In [2]:
# Load the shared helpers. vectara_utils.py sits next to this notebook in the repo;
# Colab fetches it on first run so the same notebook works locally and in Colab.
try:
    from vectara_utils import delete_and_create_agent, create_session, send_event
except ImportError:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/vectara/example-notebooks/main/notebooks/api-examples/vectara_utils.py",
        "vectara_utils.py",
    )
    from vectara_utils import delete_and_create_agent, create_session, send_event

import vectara_utils
vectara_utils.configure(BASE_URL, headers)

In [3]:
def create_corpus(corpus_key, name, documents):
    """(Re)create a corpus and index a list of (doc_id, text) documents."""
    requests.delete(f"{BASE_URL}/corpora/{corpus_key}", headers=headers)  # leftover from a previous run
    response = requests.post(f"{BASE_URL}/corpora", headers=headers, json={"key": corpus_key, "name": name})
    response.raise_for_status()
    for doc_id, text in documents:
        response = requests.post(
            f"{BASE_URL}/corpora/{corpus_key}/documents",
            headers=headers,
            json={"id": doc_id, "type": "core", "document_parts": [{"text": text}]},
        )
        response.raise_for_status()
    print(f"Created corpus '{corpus_key}' with {len(documents)} documents")


def wait_until_searchable(corpus_key, query_headers, timeout=60):
    """Newly indexed documents take a few seconds to become searchable."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        response = requests.post(
            f"{BASE_URL}/corpora/{corpus_key}/query", headers=query_headers, json={"query": "policy"}
        )
        if response.ok and response.json().get("search_results"):
            return
        time.sleep(3)
    raise TimeoutError(f"{corpus_key} not searchable after {timeout}s")

## Step 1: Acme's Own Corpus

This corpus lives in the agent's account, so the agent searches it with a regular `corpora_search` tool.

In [4]:
ACME_CORPUS_KEY = "tutorial-xt-acme-kb"

create_corpus(ACME_CORPUS_KEY, "Acme support policies", [
    ("acme-returns-process",
     "Acme return process for resold hardware: the customer must first open a support ticket with Acme "
     "and include the order number. Acme then requests the RMA number from the manufacturer on the "
     "customer's behalf. Any manufacturer restocking fee is passed through to the customer unchanged."),
    ("acme-support-hours",
     "Acme support is available Monday to Friday, 8am to 6pm Eastern time. Tickets opened outside "
     "these hours are answered the next business day."),
])
wait_until_searchable(ACME_CORPUS_KEY, headers)

Created corpus 'tutorial-xt-acme-kb' with 2 documents


## Step 2: Simulate the Partner (Globex)

In the real cross-account setup, **Globex** runs this step in **their** account and sends you two values: a corpus key and an API key.

The key is created with a single corpus role, `viewer` on Globex's corpus, and no account-level roles. `POST /v2/api_keys` returns the secret value only once, in `secret_key`.

In [5]:
PARTNER_CORPUS_KEY = "tutorial-xt-globex-kb"
PARTNER_KEY_NAME = "tutorial-xt-globex-query-key"

create_corpus(PARTNER_CORPUS_KEY, "Globex product and policy KB", [
    ("globex-returns",
     "Globex return policy: hardware can be returned within 45 days of delivery with an RMA number. "
     "Opened software is non-refundable. A restocking fee of 12 percent applies to items returned "
     "more than 30 days after delivery."),
    ("globex-warranty",
     "Globex warranty: the X200 router has a 3-year limited warranty covering hardware defects. "
     "The warranty does not cover damage from power surges or unauthorized firmware."),
])

# Delete keys left over from a previous run (API keys aren't unique by name).
response = requests.get(f"{BASE_URL}/api_keys", headers=headers, params={"limit": 100})
response.raise_for_status()
for key in response.json()["api_keys"]:
    if key["name"] == PARTNER_KEY_NAME:
        requests.delete(f"{BASE_URL}/api_keys/{key['id']}", headers=headers)

response = requests.post(
    f"{BASE_URL}/api_keys",
    headers=headers,
    json={
        "name": PARTNER_KEY_NAME,
        "corpus_roles": [{"corpus_key": PARTNER_CORPUS_KEY, "role": "viewer"}],
    },
)
response.raise_for_status()
partner_key = response.json()
PARTNER_API_KEY = partner_key["secret_key"]  # what Globex would send you; never print it
print(f"Created API key {partner_key['id']} with corpus roles {partner_key['corpus_roles']}")

partner_headers = {"x-api-key": PARTNER_API_KEY, "Content-Type": "application/json"}
wait_until_searchable(PARTNER_CORPUS_KEY, partner_headers)

Created corpus 'tutorial-xt-globex-kb' with 2 documents


Created API key aky_107 with corpus roles [{'corpus_key': 'tutorial-xt-globex-kb', 'role': 'viewer'}]


Before giving this key to an agent, check that it can do exactly one thing: query Globex's corpus. It must not be able to read Acme's corpus or write to Globex's.

In [6]:
checks = [
    ("query Globex corpus", "POST", f"{BASE_URL}/corpora/{PARTNER_CORPUS_KEY}/query", {"query": "return policy"}),
    ("query Acme corpus", "POST", f"{BASE_URL}/corpora/{ACME_CORPUS_KEY}/query", {"query": "return policy"}),
    ("add a Globex document", "POST", f"{BASE_URL}/corpora/{PARTNER_CORPUS_KEY}/documents",
     {"id": "x", "type": "core", "document_parts": [{"text": "x"}]}),
]
for label, method, url, body in checks:
    response = requests.request(method, url, headers=partner_headers, json=body)
    print(f"{label:<24} -> {response.status_code}")

query Globex corpus      -> 200
query Acme corpus        -> 403


add a Globex document    -> 403


## Step 3: Define the Cross-Tenant Search Tool

The tool is a `web_get` tool named after what it does, `search_globex_kb`. Each part of the configuration has one job:

| Field | Value | Why |
|---|---|---|
| `description_template` | What the tool searches, and that `body` is the question | This is the description the LLM sees for a `web_get` tool |
| `argument_override.url`, `.method` | Globex's query endpoint, `POST` | The credential can only be sent here |
| `argument_override.auth` | `x-api-key` header from `agent.secrets.globex_api_key` | The platform adds the header at call time and masks it in events |
| `argument_override` (the rest) | Content type, timeouts, size limits | Nothing for the LLM to get wrong |
| `input_transform` | Wraps `body` into `{"query": ..., "search": ..., "generation": ...}` | The LLM writes plain text; the API gets valid JSON |
| `output_transform` | Keeps `summary` and each result's `document_id` and `text` | The model doesn't see the raw HTTP envelope |

A note on the jq in `input_transform`: it receives `{args, agent, session, ...}`, and its output replaces the tool's arguments. In `.args | .body = (...)`, the expression on the right of `=` runs against `.args`, so the LLM's question is `.body` there, not `.args.body`.

If Globex's corpus is in a different account than the agent, nothing in this configuration changes. The URL and key are the partner's, and that's all.

In [7]:
GLOBEX_QUERY_URL = f"{BASE_URL}/corpora/{PARTNER_CORPUS_KEY}/query"

search_globex_kb = {
    "type": "web_get",
    "description_template": (
        "Search Globex's product and policy knowledge base (warranties, returns, product specs). "
        "Globex is the hardware manufacturer; its knowledge base is hosted in Globex's own Vectara account. "
        "Set `body` to the question as plain text (not JSON). "
        "The URL, method and credentials are fixed by the platform."
    ),
    "argument_override": {
        "url": GLOBEX_QUERY_URL,
        "method": "POST",
        "headers": {"Content-Type": "application/json", "Accept": "application/json"},
        "auth": {
            "type": "header",
            "header": "x-api-key",
            "value": {"$ref": "agent.secrets.globex_api_key"},
        },
        "timeout_seconds": 30,
        "follow_redirects": False,
        "head_lines": 400,
        "tail_lines": 0,
        "max_content_bytes": 65536,
    },
    "input_transform": (
        ".args | .body = ({query: .body, search: {limit: 5}, "
        "generation: {max_used_search_results: 5}} | tojson)"
    ),
    "output_transform": (
        ".content | fromjson | {answer: .summary, "
        "sources: [.search_results[] | {document_id, text}]}"
    ),
}

## Step 4: Create the Agent and Store the Partner Key as a Secret

The agent has two tools: `search_acme_kb` (`corpora_search` in its own account) and `search_globex_kb` (the `web_get` tool above).

Secrets are set after the agent exists, with `PATCH /v2/agents/{agent_key}/secrets`. Reading them back always returns `****`.

In [8]:
AGENT_NAME = "Tutorial Cross-Tenant Support"

instructions = """You are Acme's support assistant. Acme resells Globex hardware.
- Use `search_acme_kb` for Acme's own processes (support tickets, how returns are handled, support hours).
- Use `search_globex_kb` for Globex product facts and Globex policies (return window, restocking fees, warranty).
If a question involves both, call both tools. Answer only from tool results, and say which company
each fact comes from (Acme or Globex)."""

agent_config = {
    "name": AGENT_NAME,
    "description": "Answers from Acme's corpus and, across accounts, from Globex's corpus via web_get.",
    "model": {"name": "gpt-4o"},
    "first_step_name": "main",
    "steps": {
        "main": {
            "instructions": [{"type": "inline", "name": "support_instructions", "template": instructions}],
            "output_parser": {"type": "default"},
        }
    },
    "tool_configurations": {
        "search_acme_kb": {
            "type": "corpora_search",
            "description_template": "Search Acme's internal support policies and processes.",
            "query_configuration": {
                "search": {"corpora": [{"corpus_key": ACME_CORPUS_KEY}], "limit": 5},
                "generation": {"enabled": True},
            },
        },
        "search_globex_kb": search_globex_kb,
    },
}
agent_key = delete_and_create_agent(agent_config, AGENT_NAME)

response = requests.patch(
    f"{BASE_URL}/agents/{agent_key}/secrets",
    headers=headers,
    json={"secrets": {"globex_api_key": PARTNER_API_KEY}},
)
response.raise_for_status()

response = requests.get(f"{BASE_URL}/agents/{agent_key}/secrets", headers=headers)
print("Secrets as stored:", response.json())

Created agent 'Tutorial Cross-Tenant Support' (key: agt_tutorial_cross-tenant_support_a230)


Secrets as stored: {'secrets': {'globex_api_key': '****'}}


## Step 5: Ask a Question Only Globex's Corpus Can Answer

`ask()` prints the trace for each tool call:

- `tool_input`: the arguments the **LLM** chose. For `search_globex_kb` that's just `body`.
- `argument_override_paths`: which arguments the **configuration** supplied instead of the LLM.
- `resolved_argument_overrides` (on `tool_output`): the values the platform actually used, with secrets masked as `****`.
- `tool_output`: the result after `output_transform`, which is all the LLM sees.

In [9]:
def ask(question, session_key, preview_chars=600):
    """Send a question, print each tool call's audit trail, and return the answer."""
    events = send_event(agent_key, session_key, [{"type": "text", "content": question}])
    print(f"User: {question}\n")
    for event in events:
        name = event.get("tool_configuration_name")
        if event["type"] == "tool_input":
            print(f"[{name}] tool_input:                  {json.dumps(event['tool_input'])}")
            if event.get("argument_override_paths"):
                print(f"[{name}] argument_override_paths:     {event['argument_override_paths']}")
        elif event["type"] == "tool_output":
            if event.get("resolved_argument_overrides"):
                auth = {k: v for k, v in event["resolved_argument_overrides"].items() if k.startswith("auth")}
                print(f"[{name}] resolved overrides (auth):   {auth}")
            status = "ERROR " if event.get("error") else ""
            print(f"[{name}] tool_output: {status}{json.dumps(event['tool_output'])[:preview_chars]}\n")
        elif event["type"] == "error":
            print(f"[error] {event.get('messages')}")
    answer = next((e["content"] for e in events if e["type"] == "agent_output"), None)
    print(f"Agent: {answer}")
    return events


session_key = create_session(agent_key)
events = ask("What's the warranty on the Globex X200 router, and what doesn't it cover?", session_key)

User: What's the warranty on the Globex X200 router, and what doesn't it cover?

[search_globex_kb] tool_input:                  {"body": "Globex X200 router warranty and exclusions"}
[search_globex_kb] argument_override_paths:     ['url', 'method', 'headers.Content-Type', 'headers.Accept', 'follow_redirects', 'timeout_seconds', 'head_lines', 'tail_lines', 'max_content_bytes', 'response_mode', 'auth.type', 'auth.header', 'auth.value']
[search_globex_kb] resolved overrides (auth):   {'auth.type': 'header', 'auth.header': 'x-api-key', 'auth.value': '****'}
[search_globex_kb] tool_output: {"answer": "The Globex X200 router comes with a 3-year limited warranty that covers hardware defects. However, the warranty does not cover damage resulting from power surges or unauthorized firmware modifications [1].", "sources": [{"document_id": "globex-warranty", "text": "Globex warranty: the X200 router has a 3-year limited warranty covering hardware defects. The warranty does not cover damage from p

The LLM supplied only the question. The URL, method, headers, and credential came from the configuration, and the audit trail records that the `x-api-key` header was set (`auth.value`) without recording its value.

## Step 6: A Question That Needs Both Accounts

Returning a Globex router bought through Acme involves Acme's process (open a ticket, Acme gets the RMA, fees are passed through) and Globex's policy (45-day window, 12% restocking fee after 30 days). The agent queries its own corpus and Globex's corpus in the same turn and combines them.

In [10]:
session_key = create_session(agent_key)
events = ask(
    "A customer bought a Globex X200 router through us 40 days ago and wants to return it. "
    "What do they need to do, and will it cost them anything?",
    session_key,
)

User: A customer bought a Globex X200 router through us 40 days ago and wants to return it. What do they need to do, and will it cost them anything?

[search_acme_kb] tool_input:                  {"query": "return process for Globex products"}
[search_globex_kb] tool_input:                  {"body": "Globex X200 router return policy, restocking fee, return window"}
[search_globex_kb] argument_override_paths:     ['url', 'method', 'headers.Content-Type', 'headers.Accept', 'follow_redirects', 'timeout_seconds', 'head_lines', 'tail_lines', 'max_content_bytes', 'response_mode', 'auth.type', 'auth.header', 'auth.value']
[search_acme_kb] tool_output: {"summary": "I do not have enough information to answer the question about the return process for Globex products.", "response_language": "eng", "search_results": [{"score": 0.3869951069355011, "document_metadata": {}, "document_id": "acme-returns-process", "request_corpora_index": 0, "corpus_key": "tutorial-xt-acme-kb", "part_metadata": {}, "te

## Security Notes

- **Pin `url` whenever `auth` is set.** If the LLM could choose the URL, a prompt injection could send the request, and the credential with it, to a host of the attacker's choosing. Pinning `url` and `method` means the key can only be used for this one query endpoint.
- **Scope the key, not just the tool.** The partner key can only query one corpus (Step 2). Even if the agent configuration were wrong, the key couldn't read or change anything else.
- **Rotate without touching the agent.** To rotate the key, `PATCH` the secret. The tool configuration refers to `agent.secrets.globex_api_key`, not the value.
- **Use `auth`, not a pinned `headers` entry, for credentials.** `argument_override.headers` replaces whatever headers the LLM supplies, while `auth` adds the credential header alongside them. `auth` also supports `bearer`, OAuth client-credentials and token exchange, and AWS SigV4, if the remote API isn't Vectara.

## Cleanup (Optional)

Delete the agent, the partner API key, and both corpora.

In [11]:
response = requests.delete(f"{BASE_URL}/agents/{agent_key}", headers=headers)
print(f"Delete agent {agent_key}: {response.status_code}")

response = requests.delete(f"{BASE_URL}/api_keys/{partner_key['id']}", headers=headers)
print(f"Delete API key {partner_key['id']}: {response.status_code}")

for corpus_key in (ACME_CORPUS_KEY, PARTNER_CORPUS_KEY):
    response = requests.delete(f"{BASE_URL}/corpora/{corpus_key}", headers=headers)
    print(f"Delete corpus {corpus_key}: {response.status_code}")

Delete agent agt_tutorial_cross-tenant_support_a230: 204
Delete API key aky_107: 204


Delete corpus tutorial-xt-acme-kb: 204


Delete corpus tutorial-xt-globex-kb: 204


## Summary

- `corpora_search` searches corpora in the agent's own account. To query a corpus in another Vectara account, call that account's query API with a `web_get` tool.
- `argument_override.auth` with an `agent.secrets` `$ref` injects the partner's API key at call time; events show `****`.
- Pinning `url` and `method` restricts the credential to one endpoint.
- `input_transform` turns the LLM's plain-text question into the query API's JSON body, and `output_transform` reduces the response to the answer and its sources.
- For `web_get` tools, the description the LLM sees comes from `description_template`.